In [1]:
!pip install langgraph

In [2]:
!pip install langchain

In [3]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

In [4]:
pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 3.5 MB/s eta 0:00:00


In [5]:
pip install langchain-groq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.4 MB/s eta 0:00:00
  Attempting uninstall: groq
    Found existing installation: groq 1.1.1
    Uninstalling groq-1.1.1:
      Successfully uninstalled groq-1.1.1


In [28]:
from google.colab import userdata
import os

groq_api_key = userdata.get('GROQ_API_KEY')
os.environ["GROQ_API_KEY"] = groq_api_key


In [29]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


In [30]:
class state(TypedDict):
  prompt:str
  response:str

In [34]:
def llm_call(s:state)->state:
  text=s['prompt']
  prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that answers the question by user"),
    ("user", "{text}")])
  llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0.7)
  output_parser = StrOutputParser()
  chain = prompt | llm | output_parser
  result = chain.invoke({"text": text})
  s['response']=result
  return (s)

In [35]:
graph=StateGraph(state)
graph.add_node("llm_call",llm_call)
graph.add_edge(START,"llm_call")
graph.add_edge("llm_call",END)
workflow=graph.compile()


In [18]:
with open("graph.png", "wb") as f:
    f.write(workflow.get_graph().draw_mermaid_png())

In [36]:
llm_prompt="where is bombay?"
inital_state={'prompt':llm_prompt}
final_state=workflow.invoke(inital_state)
print(final_state)

{'prompt': 'where is bombay?', 'response': 'Bombay is the former name of the city now known as Mumbai, which is located in the state of Maharashtra, India. It is situated on the west coast of India, along the Arabian Sea, and is the largest city in the country. Mumbai is a major port city and is often referred to as the "financial capital" of India.\n\nThe name "Bombay" was used until 1995, when the city was officially renamed Mumbai. The name change was made to reflect the city\'s cultural and linguistic heritage, as "Mumbai" is derived from the name of the local goddess Mumbadevi.\n\nSo, to answer your question, Bombay is now known as Mumbai, and it is located in the state of Maharashtra, India.'}
